# UdaPlay - Part 2: Agent Implementation

In this notebook, we build the full UdaPlay AI Research Agent that:

1. Answers user questions using a two-tier retrieval system (RAG + Web Search)
2. Implements three core tools: `retrieve_game`, `evaluate_retrieval`, `game_web_search`
3. Uses a state machine for agent workflow management
4. Maintains conversation state across multiple queries
5. Generates clean, structured, and well-cited answers

## 1. Environment Setup

In [1]:
# Load environment variables
from dotenv import load_dotenv
import os

load_dotenv("config.env")

assert os.getenv("OPENAI_API_KEY") is not None, "OPENAI_API_KEY not found"
assert os.getenv("TAVILY_API_KEY") is not None, "TAVILY_API_KEY not found"

OPENAI_BASE_URL = os.getenv(
    "OPENAI_BASE_URL",
    "https://openai.vocareum.com/v1"
)

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [2]:
# Import required libraries
import json
import hashlib
from enum import Enum
from typing import List, Dict, Optional, Any
from datetime import datetime

import chromadb
from openai import OpenAI
from pydantic import BaseModel, Field
from tavily import TavilyClient

print("Libraries imported.")

Libraries imported.


## 2. Reconnect to Vector Store

We reconnect to the ChromaDB vector store created in Part 1.

In [3]:
class VectorStoreManager:
    """Manages the ChromaDB vector store for game information."""
    
    def __init__(self, persist_directory: str = "./chroma_db", collection_name: str = "udaplay_games"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        
        self.openai_client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=OPENAI_BASE_URL
        )
        
        self.chroma_client = chromadb.PersistentClient(path=persist_directory)
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        
        print(f"Connected to vector store. Collection '{collection_name}' has {self.collection.count()} documents.")
    
    def get_embeddings(self, texts: List[str]) -> List[List[float]]:
        response = self.openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )
        return [item.embedding for item in response.data]
    
    def add_documents(self, documents: List[str], metadatas: List[Dict], ids: List[str]):
        embeddings = self.get_embeddings(documents)
        self.collection.upsert(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )
    
    def search(self, query: str, n_results: int = 3, filter_type: Optional[str] = None) -> Dict:
        query_embedding = self.get_embeddings([query])[0]
        
        query_params = {
            "query_embeddings": [query_embedding],
            "n_results": n_results,
            "include": ["documents", "metadatas", "distances"]
        }
        
        if filter_type:
            query_params["where"] = {"type": filter_type}
        
        results = self.collection.query(**query_params)
        
        return {
            "documents": results["documents"][0] if results["documents"] else [],
            "metadatas": results["metadatas"][0] if results["metadatas"] else [],
            "distances": results["distances"][0] if results["distances"] else []
        }

# Initialize the vector store (will use existing data from Part 1)
vector_store = VectorStoreManager(
    persist_directory="./chroma_db",
    collection_name="udaplay_games"
)

Connected to vector store. Collection 'udaplay_games' has 0 documents.


In [4]:
# If the vector store is empty (Part 1 not run yet), populate it
if vector_store.collection.count() == 0:
    print("Vector store is empty. Loading data...")
    
    # Load game data
    with open("data/games/games.json", 'r') as f:
        games_data = json.load(f)
    with open("data/games/companies.json", 'r') as f:
        companies_data = json.load(f)
    
    documents = []
    metadatas = []
    ids = []
    
    for game in games_data:
        platforms_str = ", ".join(game["platforms"])
        doc = (
            f"Game Title: {game['title']}\n"
            f"Developer: {game['developer']}\n"
            f"Publisher: {game['publisher']}\n"
            f"Release Date: {game['release_date']}\n"
            f"Platforms: {platforms_str}\n"
            f"Genre: {game['genre']}\n"
            f"Description: {game['description']}"
        )
        documents.append(doc)
        metadatas.append({
            "type": "game",
            "title": game["title"],
            "developer": game["developer"],
            "publisher": game["publisher"],
            "release_date": game["release_date"],
            "genre": game["genre"]
        })
        ids.append(f"game_{hashlib.md5(game['title'].encode()).hexdigest()}")
    
    for company in companies_data:
        franchises_str = ", ".join(company["notable_franchises"])
        doc = (
            f"Company Name: {company['name']}\n"
            f"Headquarters: {company['headquarters']}\n"
            f"Founded: {company['founded']}\n"
            f"Notable Franchises: {franchises_str}\n"
            f"Current Projects: {company['current_projects']}\n"
            f"Description: {company['description']}"
        )
        documents.append(doc)
        metadatas.append({
            "type": "company",
            "name": company["name"],
            "headquarters": company["headquarters"],
            "founded": company["founded"]
        })
        ids.append(f"company_{hashlib.md5(company['name'].encode()).hexdigest()}")
    
    vector_store.add_documents(documents, metadatas, ids)
    print(f"Loaded {len(documents)} documents into vector store.")
else:
    print(f"Vector store already contains {vector_store.collection.count()} documents. Ready to use.")

Vector store is empty. Loading data...


Loaded 25 documents into vector store.


## 3. Define Agent State Machine

The agent uses a state machine to manage its workflow:

0. **REWRITE** - Resolve pronouns/references in follow-up queries using conversation history
1. **RETRIEVE** - Search internal knowledge base
2. **EVALUATE** - Assess quality of retrieved information
3. **WEB_SEARCH** - Fall back to web search if needed
4. **GENERATE** - Create final structured answer
5. **COMPLETE** - Return the answer

In [5]:
class AgentState(Enum):
    """States in the agent's workflow state machine."""
    RETRIEVE = "retrieve"
    EVALUATE = "evaluate"
    WEB_SEARCH = "web_search"
    GENERATE = "generate"
    COMPLETE = "complete"

class RetrievalResult(BaseModel):
    """Model for retrieval results."""
    documents: List[str] = Field(default_factory=list)
    metadatas: List[Dict] = Field(default_factory=list)
    distances: List[float] = Field(default_factory=list)
    source: str = "internal_db"

class EvaluationResult(BaseModel):
    """Model for evaluation results."""
    is_sufficient: bool
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str
    needs_web_search: bool

class AgentResponse(BaseModel):
    """Model for the final agent response."""
    answer: str
    sources: List[str] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    tools_used: List[str] = Field(default_factory=list)
    reasoning_steps: List[str] = Field(default_factory=list)

print("Agent state models ready.")

Agent state models ready.


## 4. Implement Agent Tools

The agent has three core tools:
1. **retrieve_game** - Search the vector database for game/company info
2. **evaluate_retrieval** - Assess quality of retrieved results
3. **game_web_search** - Search the web using Tavily API

In [6]:
class AgentTools:
    """Collection of tools available to the UdaPlay agent."""
    
    def __init__(self, vector_store: VectorStoreManager):
        self.vector_store = vector_store
        self.openai_client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=OPENAI_BASE_URL
        )
        self.tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    
    def retrieve_game(self, query: str, n_results: int = 3) -> RetrievalResult:
        """
        Tool 1: Retrieve game information from the internal vector database.
        
        Searches the ChromaDB vector store for relevant game or company information
        based on semantic similarity to the query.
        
        Args:
            query: Natural language query about a game or company
            n_results: Number of results to retrieve
            
        Returns:
            RetrievalResult with documents, metadata, and similarity scores
        """
        print(f"  [Tool: retrieve_game] Searching for: '{query}'")
        
        results = self.vector_store.search(query, n_results=n_results)
        
        retrieval = RetrievalResult(
            documents=results["documents"],
            metadatas=results["metadatas"],
            distances=results["distances"],
            source="internal_db"
        )
        
        print(f"  [Tool: retrieve_game] Found {len(retrieval.documents)} results")
        if retrieval.distances:
            best_similarity = 1 - retrieval.distances[0]
            print(f"  [Tool: retrieve_game] Best similarity score: {best_similarity:.4f}")
        
        return retrieval
    
    def evaluate_retrieval(self, query: str, retrieval: RetrievalResult) -> EvaluationResult:
        """
        Tool 2: Evaluate the quality of retrieved results.
        
        Uses an LLM to assess whether the retrieved information is sufficient
        to answer the user's query with high confidence.
        
        Args:
            query: The original user query
            retrieval: The retrieval results to evaluate
            
        Returns:
            EvaluationResult with confidence score and recommendation
        """
        print(f"  [Tool: evaluate_retrieval] Evaluating quality of retrieved results...")
        
        # Check if we have any results
        if not retrieval.documents:
            return EvaluationResult(
                is_sufficient=False,
                confidence=0.0,
                reasoning="No documents retrieved from the database.",
                needs_web_search=True
            )
        
        # Use the LLM to evaluate relevance
        context = "\n\n---\n\n".join(retrieval.documents[:3])
        
        evaluation_prompt = f"""You are an evaluation system. Assess whether the following retrieved context 
contains sufficient information to answer the user's question accurately and completely.

User Question: {query}

Retrieved Context:
{context}

Evaluate and respond in this exact JSON format:
{{
    "is_sufficient": true/false,
    "confidence": 0.0 to 1.0,
    "reasoning": "explanation of your assessment",
    "needs_web_search": true/false
}}

Rules:
- confidence >= 0.7 means the context is sufficient
- confidence < 0.7 means web search is recommended
- Consider whether the specific information asked for is explicitly present in the context
- If the question asks about something not covered in the context, set needs_web_search to true"""

        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a precise evaluation system. Always respond with valid JSON."},
                {"role": "user", "content": evaluation_prompt}
            ],
            temperature=0.1
        )
        
        try:
            eval_text = response.choices[0].message.content.strip()
            # Handle potential markdown code block wrapping
            if eval_text.startswith("```"):
                eval_text = eval_text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            eval_data = json.loads(eval_text)
            result = EvaluationResult(**eval_data)
        except (json.JSONDecodeError, Exception) as e:
            # Fallback: use distance-based heuristic
            best_similarity = 1 - retrieval.distances[0] if retrieval.distances else 0
            result = EvaluationResult(
                is_sufficient=best_similarity >= 0.7,
                confidence=best_similarity,
                reasoning=f"Fallback evaluation based on similarity score: {best_similarity:.4f}",
                needs_web_search=best_similarity < 0.7
            )
        
        print(f"  [Tool: evaluate_retrieval] Confidence: {result.confidence:.2f}, Sufficient: {result.is_sufficient}")
        print(f"  [Tool: evaluate_retrieval] Reasoning: {result.reasoning}")
        
        return result
    
    def game_web_search(self, query: str, max_results: int = 5) -> Dict[str, Any]:
        """
        Tool 3: Search the web for game information using Tavily API.
        
        Falls back to web search when internal knowledge is insufficient.
        
        Args:
            query: Search query string
            max_results: Maximum number of results to return
            
        Returns:
            Dictionary with search results including titles, URLs, and content
        """
        print(f"  [Tool: game_web_search] Searching web for: '{query}'")
        
        try:
            response = self.tavily_client.search(
                query=query,
                max_results=max_results,
                search_depth="basic",
                include_answer=True
            )
            
            results = {
                "answer": response.get("answer", ""),
                "sources": [],
                "raw_content": []
            }
            
            for result in response.get("results", []):
                results["sources"].append({
                    "title": result.get("title", ""),
                    "url": result.get("url", ""),
                    "content": result.get("content", "")
                })
                results["raw_content"].append(result.get("content", ""))
            
            print(f"  [Tool: game_web_search] Found {len(results['sources'])} web results")
            return results
            
        except Exception as e:
            print(f"  [Tool: game_web_search] Error: {str(e)}")
            return {
                "answer": "",
                "sources": [],
                "raw_content": [],
                "error": str(e)
            }

print("AgentTools ready: retrieve_game, evaluate_retrieval, game_web_search")

AgentTools ready: retrieve_game, evaluate_retrieval, game_web_search


## 5. Build the UdaPlay Agent

The agent class implements a state machine that orchestrates the tools and manages conversation history.

In [7]:
class UdaPlayAgent:
    """
    UdaPlay AI Research Agent for video game information.
    
    Implements a stateful agent with a state machine workflow:
    REWRITE -> RETRIEVE -> EVALUATE -> (WEB_SEARCH if needed) -> GENERATE -> COMPLETE
    
    Features:
    - Two-tier information retrieval (RAG + Web Search)
    - Quality evaluation of retrieved information
    - Persistent conversation memory
    - Structured, cited responses
    """
    
    def __init__(self, vector_store: VectorStoreManager):
        """
        Initialize the UdaPlay agent.
        
        Args:
            vector_store: VectorStoreManager instance for RAG
        """
        self.tools = AgentTools(vector_store)
        self.vector_store = vector_store
        self.openai_client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=OPENAI_BASE_URL
        )
        
        # Conversation history for stateful interaction
        self.conversation_history: List[Dict[str, str]] = []
        
        # Long-term memory: stores information learned from web searches
        self.memory: List[Dict[str, Any]] = []
        
        # Agent state tracking
        self.current_state = AgentState.RETRIEVE
        self.state_history: List[str] = []
        
        print("UdaPlay Agent initialized and ready!")
    
    def _transition_state(self, new_state: AgentState):
        """Transition the agent to a new state."""
        self.state_history.append(f"{self.current_state.value} -> {new_state.value}")
        self.current_state = new_state
    
    def _rewrite_query(self, query: str) -> str:
        """
        Rewrite a query to resolve pronouns and references using conversation history.
        
        If the query contains references like 'that game', 'the last one', etc.,
        this method uses the LLM to produce a self-contained query that can be
        used for retrieval without conversation context.
        
        Args:
            query: The raw user query (may contain unresolved references)
            
        Returns:
            A rewritten, self-contained query suitable for vector search
        """
        if not self.conversation_history:
            return query
        
        # Build recent conversation context
        recent = self.conversation_history[-6:]  # Last 3 exchanges
        history_str = "\n".join(
            f"{msg['role'].capitalize()}: {msg['content'][:200]}" for msg in recent
        )
        
        rewrite_prompt = f"""Given the conversation history below, rewrite the user's latest query 
into a fully self-contained search query that resolves all pronouns and references 
(e.g., 'that game', 'the last one', 'its developer') into explicit names.

Conversation history:
{history_str}

Latest query: {query}

Rewritten query (return ONLY the rewritten query, nothing else):"""
        
        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You rewrite queries to be self-contained. Output only the rewritten query."},
                {"role": "user", "content": rewrite_prompt}
            ],
            temperature=0.0
        )
        
        rewritten = response.choices[0].message.content.strip()
        if rewritten and rewritten != query:
            print(f"  [Query Rewrite] '{query}' -> '{rewritten}'")
            return rewritten
        return query
    
    def _persist_to_memory(self, query: str, answer: str, source: str):
        """
        Persist information to long-term memory.
        Useful when web search provides new information not in the vector store.
        """
        memory_entry = {
            "query": query,
            "answer": answer,
            "source": source,
            "timestamp": datetime.now().isoformat()
        }
        self.memory.append(memory_entry)
        
        # Also add to vector store for future retrieval
        doc = f"Query: {query}\nAnswer: {answer}\nSource: {source}"
        doc_id = f"memory_{hashlib.md5(query.encode()).hexdigest()}"
        self.vector_store.add_documents(
            documents=[doc],
            metadatas=[{"type": "memory", "query": query, "source": source}],
            ids=[doc_id]
        )
    
    def _generate_answer(self, query: str, context: str, sources: List[str]) -> str:
        """
        Generate a structured answer using the LLM.
        
        Args:
            query: The user's question
            context: Combined context from retrieval and/or web search
            sources: List of source descriptions
            
        Returns:
            Generated answer string
        """
        # Include conversation history for context
        history_context = ""
        if self.conversation_history:
            recent_history = self.conversation_history[-6:]  # Last 3 exchanges
            history_context = "\nPrevious conversation:\n"
            for msg in recent_history:
                history_context += f"{msg['role'].capitalize()}: {msg['content']}\n"
        
        sources_str = "\n".join(f"- {s}" for s in sources)
        
        system_prompt = """You are UdaPlay, an expert gaming research assistant. 
Generate clear, well-structured answers about video games.

Rules:
1. Always cite your sources at the end of the answer
2. Be specific and factual
3. If information comes from multiple sources, synthesize it clearly
4. Format the answer in a readable way
5. If you're not fully confident, indicate the level of certainty"""
        
        user_prompt = f"""{history_context}
Information Sources:
{sources_str}

Context:
{context}

User Question: {query}

Please provide a comprehensive, well-cited answer."""
        
        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.3
        )
        
        return response.choices[0].message.content
    
    def process_query(self, query: str) -> AgentResponse:
        """
        Process a user query through the agent's state machine.
        
        Workflow:
        1. RETRIEVE: Search internal vector database
        2. EVALUATE: Assess quality of retrieval
        3. WEB_SEARCH (conditional): Search web if retrieval insufficient
        4. GENERATE: Create structured answer
        5. COMPLETE: Return final response
        
        Args:
            query: The user's natural language question
            
        Returns:
            AgentResponse with answer, sources, confidence, and reasoning
        """
        print(f"\n{'='*60}")
        print(f"Processing Query: {query}")
        print(f"{'='*60}")
        
        # Reset state for new query
        self.current_state = AgentState.RETRIEVE
        self.state_history = []
        
        tools_used = []
        reasoning_steps = []
        sources = []
        all_context = ""
        confidence = 0.0
        
        # === QUERY REWRITING (resolve pronouns/references) ===
        search_query = self._rewrite_query(query)
        if search_query != query:
            reasoning_steps.append(f"Query rewritten: '{query}' -> '{search_query}'")
        
        # === STATE 1: RETRIEVE ===
        print(f"\n[State: {self.current_state.value.upper()}]")
        reasoning_steps.append("Step 1: Searching internal game database...")
        
        retrieval = self.tools.retrieve_game(search_query)
        tools_used.append("retrieve_game")
        
        if retrieval.documents:
            all_context = "\n\n---\n\n".join(retrieval.documents)
            sources.append("Internal Game Database (ChromaDB)")
        
        # === STATE 2: EVALUATE ===
        self._transition_state(AgentState.EVALUATE)
        print(f"\n[State: {self.current_state.value.upper()}]")
        reasoning_steps.append("Step 2: Evaluating quality of retrieved information...")
        
        evaluation = self.tools.evaluate_retrieval(search_query, retrieval)
        tools_used.append("evaluate_retrieval")
        confidence = evaluation.confidence
        
        reasoning_steps.append(f"  -> Confidence: {evaluation.confidence:.2f}")
        reasoning_steps.append(f"  -> Sufficient: {evaluation.is_sufficient}")
        reasoning_steps.append(f"  -> Reasoning: {evaluation.reasoning}")
        
        # === STATE 3: WEB_SEARCH (conditional) ===
        if evaluation.needs_web_search:
            self._transition_state(AgentState.WEB_SEARCH)
            print(f"\n[State: {self.current_state.value.upper()}]")
            reasoning_steps.append("Step 3: Internal knowledge insufficient. Searching the web...")
            
            web_results = self.tools.game_web_search(search_query)
            tools_used.append("game_web_search")
            
            if web_results.get("answer"):
                all_context += f"\n\n---\nWeb Search Results:\n{web_results['answer']}"
                sources.append("Web Search (Tavily)")
                
            if web_results.get("raw_content"):
                web_context = "\n".join(web_results["raw_content"][:3])
                all_context += f"\n\nAdditional Web Content:\n{web_context}"
            
            # Add web source URLs
            for src in web_results.get("sources", [])[:3]:
                sources.append(f"Web: {src.get('title', 'Unknown')} ({src.get('url', '')})")
            
            # Boost confidence if web search found relevant results
            if web_results.get("answer") or web_results.get("raw_content"):
                confidence = max(confidence, 0.8)
                reasoning_steps.append("  -> Web search provided additional information")
            
            # Persist web search results to memory
            if web_results.get("answer"):
                self._persist_to_memory(query, web_results["answer"], "web_search")
                reasoning_steps.append("  -> Persisted web results to long-term memory")
        else:
            reasoning_steps.append("Step 3: Internal knowledge sufficient. Skipping web search.")
        
        # === STATE 4: GENERATE ===
        self._transition_state(AgentState.GENERATE)
        print(f"\n[State: {self.current_state.value.upper()}]")
        reasoning_steps.append("Step 4: Generating structured answer...")
        
        answer = self._generate_answer(query, all_context, sources)
        
        # === STATE 5: COMPLETE ===
        self._transition_state(AgentState.COMPLETE)
        print(f"\n[State: {self.current_state.value.upper()}]")
        reasoning_steps.append("Step 5: Query processing complete.")
        
        # Update conversation history
        self.conversation_history.append({"role": "user", "content": query})
        self.conversation_history.append({"role": "assistant", "content": answer})
        
        # Build final response
        response = AgentResponse(
            answer=answer,
            sources=sources,
            confidence=confidence,
            tools_used=tools_used,
            reasoning_steps=reasoning_steps
        )
        
        return response
    
    def generate_report(self, response: AgentResponse) -> str:
        """
        Generate a clean, formatted report from an agent response.
        
        Args:
            response: The AgentResponse to format
            
        Returns:
            Formatted report string
        """
        report = []
        report.append("\n" + "=" * 60)
        report.append("UDAPLAY AGENT REPORT")
        report.append("=" * 60)
        
        report.append(f"\n--- Answer ---")
        report.append(response.answer)
        
        report.append(f"\n--- Confidence Level ---")
        confidence_bar = "#" * int(response.confidence * 20) + "-" * (20 - int(response.confidence * 20))
        report.append(f"[{confidence_bar}] {response.confidence:.0%}")
        
        report.append(f"\n--- Sources ---")
        for i, source in enumerate(response.sources, 1):
            report.append(f"  {i}. {source}")
        
        report.append(f"\n--- Tools Used ---")
        report.append(f"  {' -> '.join(response.tools_used)}")
        
        report.append(f"\n--- Reasoning Steps ---")
        for step in response.reasoning_steps:
            report.append(f"  {step}")
        
        report.append("\n" + "=" * 60)
        
        return "\n".join(report)
    
    def ask(self, query: str) -> str:
        """
        High-level method to ask the agent a question and get a formatted report.
        
        Args:
            query: Natural language question
            
        Returns:
            Formatted report string
        """
        response = self.process_query(query)
        report = self.generate_report(response)
        return report

print("UdaPlayAgent ready.")

UdaPlayAgent ready.


## 6. Initialize and Test the Agent

In [8]:
# Initialize the agent
agent = UdaPlayAgent(vector_store)

UdaPlay Agent initialized and ready!


## 7. Run Example Queries

Let's demonstrate the agent with multiple queries showing different scenarios.

In [9]:
# Query 1: A question that should be answerable from internal knowledge
report1 = agent.ask("Who developed FIFA 21?")
print(report1)


Processing Query: Who developed FIFA 21?

[State: RETRIEVE]
  [Tool: retrieve_game] Searching for: 'Who developed FIFA 21?'


  [Tool: retrieve_game] Found 3 results
  [Tool: retrieve_game] Best similarity score: 0.7340

[State: EVALUATE]
  [Tool: evaluate_retrieval] Evaluating quality of retrieved results...


  [Tool: evaluate_retrieval] Confidence: 1.00, Sufficient: True
  [Tool: evaluate_retrieval] Reasoning: The retrieved context explicitly states that FIFA 21 was developed by EA Vancouver, which directly answers the user's question.

[State: GENERATE]



[State: COMPLETE]

UDAPLAY AGENT REPORT

--- Answer ---
FIFA 21 was developed by EA Vancouver, a studio under the umbrella of Electronic Arts (EA). EA Vancouver has been responsible for several installments in the FIFA series, contributing to the franchise's reputation as a leading football simulation game. FIFA 21 was released on October 9, 2020, and is notable for its improved gameplay mechanics, enhanced artificial intelligence, and the introduction of new game modes, including an updated Career Mode and the return of FIFA Ultimate Team (source: Internal Game Database - ChromaDB).

EA Vancouver's work on FIFA 21 continues the legacy of the FIFA franchise, which has been a flagship title for EA since its inception, with the series being one of the most successful in sports gaming history (source: Internal Game Database - ChromaDB).

--- Confidence Level ---
[####################] 100%

--- Sources ---
  1. Internal Game Database (ChromaDB)

--- Tools Used ---
  retrieve_game -> eval

In [10]:
# Query 2: Another query from internal knowledge
report2 = agent.ask("When was God of War Ragnarok released and on which platforms?")
print(report2)


Processing Query: When was God of War Ragnarok released and on which platforms?


  [Query Rewrite] 'When was God of War Ragnarok released and on which platforms?' -> 'When was the video game God of War Ragnarok released and on which gaming platforms is it available?'

[State: RETRIEVE]
  [Tool: retrieve_game] Searching for: 'When was the video game God of War Ragnarok released and on which gaming platforms is it available?'


  [Tool: retrieve_game] Found 3 results
  [Tool: retrieve_game] Best similarity score: 0.6953

[State: EVALUATE]
  [Tool: evaluate_retrieval] Evaluating quality of retrieved results...


  [Tool: evaluate_retrieval] Confidence: 1.00, Sufficient: True
  [Tool: evaluate_retrieval] Reasoning: The retrieved context provides the exact release date of God of War Ragnarok (November 9, 2022) and lists the available platforms (PlayStation 4, PlayStation 5, PC), which directly answers the user's question.

[State: GENERATE]



[State: COMPLETE]

UDAPLAY AGENT REPORT

--- Answer ---
**God of War Ragnarok Release Information**

**Release Date:** God of War Ragnarok was released on **November 9, 2022**.

**Platforms:** The game is available on the following platforms:
- **PlayStation 4**
- **PlayStation 5**
- **PC**

**Overview:** Developed by Santa Monica Studio and published by Sony Interactive Entertainment, God of War Ragnarok is the ninth installment in the God of War series and serves as a sequel to the critically acclaimed God of War (2018). The game follows the journey of Kratos and his son Atreus as they navigate the Nine Realms in the face of the impending doom of Ragnarok.

This information is sourced from the official announcements and game databases that track video game releases and specifications (source: Internal Game Database - ChromaDB).

--- Confidence Level ---
[####################] 100%

--- Sources ---
  1. Internal Game Database (ChromaDB)

--- Tools Used ---
  retrieve_game -> evaluate

In [11]:
# Query 3: A query about current projects (may trigger web search)
report3 = agent.ask("What is Rockstar Games working on right now?")
print(report3)


Processing Query: What is Rockstar Games working on right now?


  [Query Rewrite] 'What is Rockstar Games working on right now?' -> 'What is Rockstar Games currently developing?'

[State: RETRIEVE]
  [Tool: retrieve_game] Searching for: 'What is Rockstar Games currently developing?'


  [Tool: retrieve_game] Found 3 results
  [Tool: retrieve_game] Best similarity score: 0.7271

[State: EVALUATE]
  [Tool: evaluate_retrieval] Evaluating quality of retrieved results...


  [Tool: evaluate_retrieval] Confidence: 0.90, Sufficient: True
  [Tool: evaluate_retrieval] Reasoning: The retrieved context explicitly states that Rockstar Games is currently developing Grand Theft Auto VI (GTA 6), which answers the user's question accurately and completely.

[State: GENERATE]



[State: COMPLETE]

UDAPLAY AGENT REPORT

--- Answer ---
**Current Projects of Rockstar Games**

As of October 2023, Rockstar Games is primarily focused on the development of **Grand Theft Auto VI (GTA 6)**. This highly anticipated title was officially announced in December 2023, with a trailer released to generate excitement among fans. The game is expected to be released in **2025**.

### Overview of Grand Theft Auto VI
- **Announcement:** December 2023
- **Expected Release Date:** 2025
- **Setting and Features:** While specific details about the game’s setting and features have not been fully disclosed, it is anticipated that GTA 6 will continue the franchise's tradition of expansive open-world gameplay, intricate narratives, and a richly detailed environment.

### Additional Information
Rockstar Games has a reputation for developing high-quality, immersive games, and GTA 6 is expected to uphold this legacy. The company is known for its meticulous attention to detail and commitment 

In [12]:
# Query 4: A query that likely requires web search (not in internal DB)
report4 = agent.ask("What platform was Pokemon Red launched on?")
print(report4)


Processing Query: What platform was Pokemon Red launched on?


  [Query Rewrite] 'What platform was Pokemon Red launched on?' -> 'What platform was Pokémon Red launched on?'

[State: RETRIEVE]
  [Tool: retrieve_game] Searching for: 'What platform was Pokémon Red launched on?'


  [Tool: retrieve_game] Found 3 results
  [Tool: retrieve_game] Best similarity score: 0.6743

[State: EVALUATE]
  [Tool: evaluate_retrieval] Evaluating quality of retrieved results...


  [Tool: evaluate_retrieval] Confidence: 1.00, Sufficient: True
  [Tool: evaluate_retrieval] Reasoning: The retrieved context explicitly states that Pokémon Red was launched on the Game Boy, which directly answers the user's question.

[State: GENERATE]



[State: COMPLETE]

UDAPLAY AGENT REPORT

--- Answer ---
**Platform Information for Pokémon Red**

**Launch Platform:** Pokémon Red was launched on the **Game Boy**.

### Overview
Pokémon Red, developed by Game Freak and published by Nintendo, was released on **February 27, 1996**. It is one of the first games in the Pokémon series and was designed for the original Game Boy handheld console. The game allows players to take on the role of a Pokémon Trainer, exploring the Kanto region, capturing and training Pokémon, and battling other trainers to become the Pokémon Champion.

### Significance
The Game Boy platform played a crucial role in the success of Pokémon Red, as it was one of the first portable gaming devices that allowed players to enjoy gaming on the go. The game's success contributed significantly to the popularity of the Pokémon franchise, which has since expanded into various media, including animated series, movies, and a wide range of merchandise.

### Conclusion
In summar

In [13]:
# Query 5: Testing conversation memory - follow up question
report5 = agent.ask("Tell me more about the developer of that last game I asked about.")
print(report5)


Processing Query: Tell me more about the developer of that last game I asked about.


  [Query Rewrite] 'Tell me more about the developer of that last game I asked about.' -> 'Tell me more about the developer of Pokémon Red.'

[State: RETRIEVE]
  [Tool: retrieve_game] Searching for: 'Tell me more about the developer of Pokémon Red.'


  [Tool: retrieve_game] Found 3 results
  [Tool: retrieve_game] Best similarity score: 0.6208

[State: EVALUATE]
  [Tool: evaluate_retrieval] Evaluating quality of retrieved results...


  [Tool: evaluate_retrieval] Confidence: 0.90, Sufficient: True
  [Tool: evaluate_retrieval] Reasoning: The retrieved context provides clear information about the developer of Pokémon Red, which is Game Freak. It includes relevant details about the game itself, its publisher, and its release date, which collectively give a comprehensive overview of the game's development.

[State: GENERATE]



[State: COMPLETE]

UDAPLAY AGENT REPORT

--- Answer ---
**Developer Overview: Game Freak**

### Company Profile
- **Name:** Game Freak, Inc.
- **Founded:** 1989
- **Headquarters:** Tokyo, Japan
- **Notable Franchises:** Pokémon series, including Pokémon Red, Pokémon Blue, Pokémon Gold and Silver, and many others.

### History and Background
Game Freak started as a fanzine dedicated to video games, created by Satoshi Tajiri and Ken Sugimori. The company transitioned into game development in the early 1990s, and its first title was **Mario & Wario** for the Super Famicom in 1993. However, Game Freak is best known for its long-standing partnership with Nintendo, particularly for the Pokémon franchise.

### Pokémon Franchise
The Pokémon series began with the release of **Pokémon Red and Green** in Japan in 1996 (with Pokémon Red and Blue later released internationally). The games introduced players to the world of Pokémon, where they could capture, train, and battle creatures. The franchi

## 8. Agent State and Memory Inspection

In [14]:
# Inspect agent's conversation history
print("=" * 60)
print("CONVERSATION HISTORY")
print("=" * 60)
for i, msg in enumerate(agent.conversation_history):
    role = msg['role'].upper()
    content = msg['content'][:100] + "..." if len(msg['content']) > 100 else msg['content']
    print(f"\n[{i+1}] {role}: {content}")

CONVERSATION HISTORY

[1] USER: Who developed FIFA 21?

[2] ASSISTANT: FIFA 21 was developed by EA Vancouver, a studio under the umbrella of Electronic Arts (EA). EA Vanco...

[3] USER: When was God of War Ragnarok released and on which platforms?

[4] ASSISTANT: **God of War Ragnarok Release Information**

**Release Date:** God of War Ragnarok was released on *...

[5] USER: What is Rockstar Games working on right now?

[6] ASSISTANT: **Current Projects of Rockstar Games**

As of October 2023, Rockstar Games is primarily focused on t...

[7] USER: What platform was Pokemon Red launched on?

[8] ASSISTANT: **Platform Information for Pokémon Red**

**Launch Platform:** Pokémon Red was launched on the **Gam...

[9] USER: Tell me more about the developer of that last game I asked about.

[10] ASSISTANT: **Developer Overview: Game Freak**

### Company Profile
- **Name:** Game Freak, Inc.
- **Founded:** ...


In [15]:
# Inspect agent's long-term memory
print("=" * 60)
print("LONG-TERM MEMORY")
print("=" * 60)
if agent.memory:
    for i, entry in enumerate(agent.memory):
        print(f"\n[Memory {i+1}]")
        print(f"  Query: {entry['query']}")
        print(f"  Source: {entry['source']}")
        print(f"  Timestamp: {entry['timestamp']}")
        print(f"  Answer: {entry['answer'][:150]}...")
else:
    print("\nNo web search results have been persisted to memory yet.")
    print("(Memory is populated when web search is triggered for insufficient internal knowledge)")

LONG-TERM MEMORY

No web search results have been persisted to memory yet.
(Memory is populated when web search is triggered for insufficient internal knowledge)


In [16]:
# Show state transition history
print("=" * 60)
print("STATE TRANSITIONS (Last Query)")
print("=" * 60)
for transition in agent.state_history:
    print(f"  {transition}")

STATE TRANSITIONS (Last Query)
  retrieve -> evaluate
  evaluate -> generate
  generate -> complete
